# 01 — Introduction : taxonomies de sophismes et terrain du cours

**Phase 1 / livrable d'intro de l'EPIC [#10355](https://github.com/jsboige/CoursIA/issues/10355)** (detection de sophismes par Qwen FT + PT). Ce notebook introduit les concepts avant le [paysage des datasets (02)](02_fallacy_datasets_landscape.ipynb) et la [mesure de l'écart de couverture (03)](03_taxonomy_coverage_gap.ipynb) — voir le [README de la série](README.md) pour la chaîne des phases.

**Plan.** (1) ce qu'est un sophisme, formel vs informel ; (2) le terrain du cours : la taxonomie Argumentum (repo-locale) ; (3) les taxonomies académiques en face — et le choix de corpus qui en découle.


## 1. Ce qu'est un sophisme

Un **sophisme** est un raisonnement *qui paraît* valide mais dont la conclusion ne suit pas des prémisses — ou ne suit que par un détour rhétorique fautif. On distingue :

- les **sophismes formels** : la forme logique elle-même est invalide (ex. *affirmation du conséquent* : « s'il pleut, le sol est mouillé ; le sol est mouillé ; donc il a plu ») ;
- les **sophismes informels** : la forme peut être correcte mais le contenu triche (ex. *ad hominem* : disqualifier l'auteur plutôt que l'argument ; *faux dilemme* : réduire à deux options ce qui en admet d'autres).

La détection automatique de sophismes (l'objet de cette série) consiste à étiqueter un texte d'argumentation avec le (ou les) sophisme(s) qu'il commet — ce qui suppose d'abord de **fixer la taxonomie des étiquettes**. C'est le rôle des sections 2 et 3.


In [1]:
# Six exemples courts, avec l'etiquette attendue (formel vs informel + type).
# Ce sont des exemples construits pour le cours, pas des extraits de corpus.
exemples = {
    "E1": "S'il pleut, le sol est mouille. Le sol est mouille. Donc il a plu.",
    "E2": "Tous les hommes sont mortels. Socrate est mortel. Donc Socrate est un homme.",
    "E3": "Mon contradicteur est un mauvais pere, donc son argument ne vaut rien.",
    "E4": "Soit tu soutiens cette reforme, soit tu es contre le progres.",
    "E5": "Ce medicament a ete pris par 3 personnes gueries, donc il soigne.",
    "E6": "Le stade est soit plein, soit vide ; il n'est pas plein, donc il est vide.",
}
attendu = {"E1": ("formel", "affirmation du consequent"),
           "E2": ("formel", "affirmation du consequent"),
           "E3": ("informel", "ad hominem"),
           "E4": ("informel", "faux dilemme"),
           "E5": ("informel", "generalisation hative"),
           "E6": ("formel", "fausse disjonction")}
for k, txt in exemples.items():
    cat, t = attendu[k]
    print(f"{k} [{cat:8s}] {t:28s} :: {txt[:60]}")


E1 [formel  ] affirmation du consequent    :: S'il pleut, le sol est mouille. Le sol est mouille. Donc il 
E2 [formel  ] affirmation du consequent    :: Tous les hommes sont mortels. Socrate est mortel. Donc Socra
E3 [informel] ad hominem                   :: Mon contradicteur est un mauvais pere, donc son argument ne 
E4 [informel] faux dilemme                 :: Soit tu soutiens cette reforme, soit tu es contre le progres
E5 [informel] generalisation hative        :: Ce medicament a ete pris par 3 personnes gueries, donc il so
E6 [formel  ] fausse disjonction           :: Le stade est soit plein, soit vide ; il n'est pas plein, don


### Exercice 1 — Formel ou informel ?

**Contexte.** Les six exemples ci-dessus sont étiquetés dans `attendu`. Avant de lire cette variable, entraînez l'œil : la distinction formel/informel est le premier geste du détecteur (humain ou modèle).

**Objectif.** Pour chaque exemple `E1`–`E6`, produire votre propre couple `(formel|informel, type)` dans `etiquettes_formel`, puis comparer avec `attendu` et compter les désaccords.


In [2]:
# Exercice 1 — Etiqueter les six exemples
# Etape 1 : pour chaque cle E1..E6, decider formel ou informel + le type.
# Indice : deux exemples seulement sont des paralogismes formiques pures (E1, E2).
# Indice : E6 cache une disjonction qui n'est pas exhaustive -- formel ou informel ?
# Etape 2 : comparer avec attendu et compter les desaccords.
etiquettes_formel = None  # TODO etudiant


## 2. Le terrain du cours : la taxonomie Argumentum

Le cours ancre la détection sur la taxonomie **Argumentum** (catalogue francophone de 1 408 nœuds de sophismes, hiérarchie jusqu'à la profondeur 4, traduite en 8 langues — le CSV repo-local expose des colonnes `_fr` et `_en` entre autres). Mesure et discussion dans [03](03_taxonomy_coverage_gap.ipynb) ; ici, on charge et on compte.


In [3]:
import csv
from pathlib import Path
from collections import Counter

# Robuste au cwd (papermill peut lancer depuis repo-root ou le dossier notebook).
_CANDIDATES = [
    Path("MyIA.AI.Notebooks/SymbolicAI/Argument_Analysis/data/argumentum_fallacies_taxonomy.csv"),
    Path("../SymbolicAI/Argument_Analysis/data/argumentum_fallacies_taxonomy.csv"),
    Path("../../SymbolicAI/Argument_Analysis/data/argumentum_fallacies_taxonomy.csv"),
]
CSV_PATH = next((c for c in _CANDIDATES if c.is_file()), None)
assert CSV_PATH is not None, "Taxonomie introuvable depuis cwd=" + str(Path.cwd())

with open(CSV_PATH, encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))
all_paths = {r["path"].strip() for r in rows if r.get("path")}
leaves = [r for r in rows
          if r["path"].strip() and r["path"].strip() != "0"
          and not any(p.startswith(r["path"].strip() + ".") for p in all_paths
                      if p != r["path"].strip())]
families = Counter(r["Famille"].strip() for r in leaves if r.get("Famille", "").strip())
print(f"Taxonomie Argumentum chargee : {CSV_PATH}")
print(f"  noeuds total : {len(rows)} | feuilles (sophismes identifiables) : {len(leaves)}")
print("  top familles :")
for fam, n in families.most_common(5):
    print(f"    {fam:<28} {n}")


Taxonomie Argumentum chargee : MyIA.AI.Notebooks\SymbolicAI\Argument_Analysis\data\argumentum_fallacies_taxonomy.csv
  noeuds total : 1408 | feuilles (sophismes identifiables) : 894
  top familles :
    Influence                    285
    Tricherie                    255
    Insuffisance                 102
    Obstruction                  77
    Erreur mathématique          61


### Exercice 2 — Compter les feuilles d'une mini-taxonomie

**Contexte.** Le comptage de feuilles ci-dessus utilise le critère de préfixe : un nœud est une **feuille** si son `path` n'est préfixe d'aucun autre. Voici une mini-taxonomie du même format :

```python
mini = {"1", "1.1", "1.1.1", "1.1.2", "1.2", "2", "2.1", "3"}
```

**Objectif.** Écrire une fonction `compte_feuilles(chemins)` qui généralise ce critère (sans réutiliser la variable `leaves`), la tester sur `mini` (attendu : 5), puis sur l'ensemble des `path` réels du CSV.


In [4]:
# Exercice 2 — Fonction compte_feuilles(chemins)
# Etape 1 : ecrire compte_feuilles(chemins) -> int par le critere de prefixe.
# Indice : un path P est feuille ssi aucun autre path ne commence par P + ".".
# Etape 2 : tester sur mini (attendu 5), puis sur all_paths du CSV.
# Indice : attention au path racine "0" -- il est exclus du comptage des feuilles dans la cellule ci-dessus.
nb_feuilles = None  # TODO etudiant


## 3. Les taxonomies académiques en face

Les corpus académiques annotés (mesurés en accès réel dans [02](02_fallacy_datasets_landscape.ipynb)) portent des taxonomies **beaucoup plus petites** — et c'est la tension centrale de la série :

| Taxonomie | Référence primaire | Classes fallacy |
|---|---|---|
| Logic / LogicClimate | Jin et al. 2022, [arXiv:2202.13758](https://arxiv.org/abs/2202.13758) | 13 |
| MAFALDA | Helwe, Calamai, Paris, Clavel 2023 | 23 (niveau L2) |
| Argotario | Habernal, Pauli & Gurevych, NAACL 2017 (demo) | ~5 |
| Walton (extra) | Walton, *A Pragmatic Theory of Fallacy*, 1995 | encyclopédique (informels) |
| **Argumentum (le terrain)** | taxonomie Argumentum, éd. 2022 (CSV repo-local) | **~1 200 feuilles** |

L'écart — plus d'un ordre de grandeur — est mesuré et discuté dans [03](03_taxonomy_coverage_gap.ipynb) ; la stratégie de données qui en découle (corpus synthétique par produit cartésien Scénarii × Fallacy) est détaillée dans le [README de la série](README.md).


In [5]:
# Table comparative chiffree (constantes du markdown ci-dessus ; cf. 02/03 pour les mesures).
taxonomies = {
    "Logic/LogicClimate (Jin et al. 2022)": 13,
    "MAFALDA L2 (Helwe et al. 2023)": 23,
    "Argotario (Habernal et al. 2017)": 5,
    "Argumentum (feuilles, CSV repo-local)": len(leaves),
}
ref = taxonomies["Logic/LogicClimate (Jin et al. 2022)"]
for nom, n in taxonomies.items():
    print(f"{nom:<42} {n:>5} classes  (x{n / ref:.0f} vs Logic)")


Logic/LogicClimate (Jin et al. 2022)          13 classes  (x1 vs Logic)
MAFALDA L2 (Helwe et al. 2023)                23 classes  (x2 vs Logic)
Argotario (Habernal et al. 2017)               5 classes  (x0 vs Logic)
Argumentum (feuilles, CSV repo-local)        894 classes  (x69 vs Logic)


### Exercice 3 — Choisir le corpus d'entraînement de la Phase 2

**Contexte.** Le README de la série pose deux voies : corpus académiques annotés (petits, humains) vs corpus synthétique par produit cartésien (couverture complète). Le choix n'est pas exclusif — mais il faut savoir **pour quel usage** chaque voie est bonne.

**Objectif.** Au vu de la table ci-dessus et des critères de [02](02_fallacy_datasets_landscape.ipynb) (labels explicites, cardinalité, accès programmatique), désigner le corpus pilote du fine-tuning Phase 3 et justifier en trois lignes : que valide-t-il, que ne valide-t-il pas ?


In [6]:
# Exercice 3 — Choix du corpus pilote (Phase 3 fine-tuning)
# Etape 1 : relire les criteres de 02 (labels fallacy explicites, cardinalite, acces).
# Indice : une evaluation OOS sur donnees humaines exige un corpus reellement annote.
# Indice : la couverture uniforme de la taxonomie n'est portee que par la voie synthetique.
# Etape 2 : designer le pilote + 3 lignes de justification (valide / ne valide pas).
choix_pilote = None  # TODO etudiant


## Aller plus loin

- [02 — Paysage des datasets](02_fallacy_datasets_landscape.ipynb) : accès réel à 7 datasets annotés.
- [03 — Écart de couverture taxonomique](03_taxonomy_coverage_gap.ipynb) : la mesure qui requalifie le paysage.
- [README de la série](README.md) : chaîne des phases (survey → builder → FT → PT → SAE).
- Survey SOTA fondateur : [docs/research/fallacy-detection-survey.md](../../../docs/research/fallacy-detection-survey.md) (10 sources primaires).
